# Question / Remark

In the assignment it said Mistral and now it changed to Anthropic.

We recognized it too late, so now we classify the sentence with the highest Mistral probability as Anthropic

## 1. Config

The notebook which trained the model is located in notebooks/logReg_tfidf.ipynb

It was trained with following config:

In [1]:
# =========================
# CONFIG
# =========================


CONFIG = {
    "size": 100000,

    "word_max_features": 15000,
    "char_max_features": 15000,

    "min_df": 2,

    "word_ngram_range": (1, 2),
    "char_ngram_range": (2, 6),

    "lr": 0.3,
    "epochs": 40,
}

print("CONFIGURATION:")
for k, v in CONFIG.items():
    print(f"{k}: {v}")

CONFIGURATION:
size: 100000
word_max_features: 15000
char_max_features: 15000
min_df: 2
word_ngram_range: (1, 2)
char_ngram_range: (2, 6)
lr: 0.3
epochs: 40


## 2. Imports

In [2]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

from src.tfidf.tfidf import TfIdfVectorizerNumpy

## 3. Load model

In [3]:
print("\nLoad model...")

def load_model(path):
    with open(path, "rb") as f:
        return pickle.load(f)

BASE_DIR = Path("..")
MODEL_PATH = Path("../models/subm1-g5-MEI-A.pkl")
model_data = load_model(MODEL_PATH)


Load model...


## 4. Rebuild vectorizers

In [4]:
print("\nRebuilding vectorizers...")

word_vectorizer = TfIdfVectorizerNumpy()
word_vectorizer.vocab = model_data["word_vocab"]
word_vectorizer.idf = model_data["word_idf"]

char_vectorizer = TfIdfVectorizerNumpy(analyzer="char")
char_vectorizer.vocab = model_data["char_vocab"]
char_vectorizer.idf = model_data["char_idf"]


Rebuilding vectorizers...


## 5 Access model and subm1.csv

In [5]:
print("\nAccess model...")

clf = model_data["model"]

print("\nLoading CSV...")

df = pd.read_csv(
    str(BASE_DIR) + "/tests/TestData/subm1.csv",
    sep=";"
)

texts = df["Text"].tolist()

print("Number of samples:", len(texts))


Access model...

Loading CSV...
Number of samples: 150


## 6. Build features

In [6]:
print("\nBuilding features for CSV...")

X_word = word_vectorizer.transform(texts)
X_char = char_vectorizer.transform(texts)

X = np.hstack([X_word, X_char])

length = np.array([len(t) for t in texts]).reshape(-1, 1) / 200
X = np.hstack([X, length])

print("CSV feature shape:", X.shape)


Building features for CSV...
CSV feature shape: (150, 30001)


## 7. Make predictions and save it

In [7]:
id_to_company = {
    0: "Human",
    1: "OpenAI",
    2: "Anthropic",
    3: "Google",
    4: "Meta"
}


print("\nRunning predictions...")

preds = clf.predict(X)

df["Label"] = [id_to_company[p] for p in preds]

print(df.head())



df[["ID", "Text", "Label"]].to_csv("subm1-g5-MEI-A.csv", index=False, seperator=";")
print("Saved: subm1-g5-MEI-A.csv")


Running predictions...
     ID                                               Text      Label
0  D2-1  A covalent bond is a chemical bond that involv...  Anthropic
1  D2-2  A covalent bond forms when two atoms share one...  Anthropic
2  D2-3  A covalent bond is a type of chemical bond whe...      Human
3  D2-4  A covalent bond is a chemical bond that involv...     OpenAI
4  D2-5  Driven by exciting developments in the field o...  Anthropic


TypeError: NDFrame.to_csv() got an unexpected keyword argument 'seperator'